In [1]:
import torch
print(torch.__version__)

2.11.0+cpu


In [2]:
from google.colab import files
uploaded = files.upload()

Saving daphnet+freezing+of+gait.zip to daphnet+freezing+of+gait.zip


In [3]:
import zipfile

with zipfile.ZipFile("daphnet+freezing+of+gait.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [4]:
import os
os.listdir()

['.config',
 'daphnet+freezing+of+gait.zip',
 'dataset_fog_release',
 'sample_data']

In [5]:
  import os
os.listdir("dataset_fog_release")

['dataset', 'doc', 'scripts', 'README']

In [6]:
import os
import re
import glob
import numpy as np
import pandas as pd

files = sorted(
    glob.glob("dataset_fog_release/dataset/*.txt")
)

print(f"Found {len(files)} files")

Found 17 files


In [7]:
WINDOW = 128
STRIDE = 64

all_windows = []
all_labels = []
all_groups = []

for file_path in files:

    patient_id = re.findall(
        r"(S\d+)",
        os.path.basename(file_path)
    )[0]

    df = pd.read_csv(
        file_path,
        sep=" ",
        header=None,
        engine="c"
    )

    df = df[df.iloc[:, -1] != -1]

    X_raw = df.iloc[:, 1:-1].values
    y_raw = (df.iloc[:, -1].values > 0).astype(int)

    for i in range(
        0,
        len(X_raw) - WINDOW,
        STRIDE
    ):

        window_x = X_raw[i:i+WINDOW]

        window_y = int(
            np.max(
                y_raw[i:i+WINDOW]
            )
        )

        all_windows.append(window_x)
        all_labels.append(window_y)
        all_groups.append(patient_id)

X = np.array(all_windows)
y = np.array(all_labels)
groups = np.array(all_groups)

print(X.shape)
print(y.shape)

(29940, 128, 9)
(29940,)


In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader



In [9]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        X,
        y,
        groups
    )
)

X_train = X[train_idx]
y_train = y[train_idx]
groups_train = groups[train_idx]

X_temp = X[temp_idx]
y_temp = y[temp_idx]
groups_temp = groups[temp_idx]

In [10]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss.split(
        X_temp,
        y_temp,
        groups_temp
    )
)

X_val = X_temp[val_idx]
y_val = y_temp[val_idx]

X_test = X_temp[test_idx]
y_test = y_temp[test_idx]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (21284, 128, 9)
Validation: (2691, 128, 9)
Test: (5965, 128, 9)


In [20]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_train_flat = X_train.reshape(
    -1,
    X_train.shape[-1]
)

scaler.fit(X_train_flat)

def transform_windows(X):

    original_shape = X.shape

    X = X.reshape(
        -1,
        X.shape[-1]
    )

    X = scaler.transform(X)

    return X.reshape(
        original_shape
    )

X_train = transform_windows(X_train)
X_val = transform_windows(X_val)
X_test = transform_windows(X_test)

joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']

In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# CNN expects (batch, channels, sequence)
X_train_cnn = np.transpose(X_train, (0, 2, 1))
X_val_cnn = np.transpose(X_val, (0, 2, 1))
X_test_cnn = np.transpose(X_test, (0, 2, 1))

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train_cnn, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    ),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val_cnn, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    ),
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test_cnn, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long)
    ),
    batch_size=32,
    shuffle=False
)

In [22]:
import torch.nn as nn

class FOGModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv1d(9, 32, 5),

            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Dropout(0.30),

            nn.Conv1d(32, 64, 5),

            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Dropout(0.30),
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.30)

        self.fc = nn.Linear(
            64,
            2
        )

    def forward(self, x):

        x = self.cnn(x)

        x = x.permute(
            0,
            2,
            1
        )

        _, (h, _) = self.lstm(x)

        h = self.dropout(h[-1])

        return self.fc(h)

model = FOGModel()

In [23]:
import torch

weights = torch.tensor([1.0, 1.5])  # penalize missing FOG more
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Avg Loss: 0.2907
Epoch 2, Avg Loss: 0.2384
Epoch 3, Avg Loss: 0.2150
Epoch 4, Avg Loss: 0.2005
Epoch 5, Avg Loss: 0.1897
Epoch 6, Avg Loss: 0.1809
Epoch 7, Avg Loss: 0.1697
Epoch 8, Avg Loss: 0.1555
Epoch 9, Avg Loss: 0.1494
Epoch 10, Avg Loss: 0.1387


In [24]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb)
        probs = torch.softmax(out, dim=1)[:, 1]   # probability of FOG
        preds = (probs > 0.4).int()

        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

In [25]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

[[1467 1079]
 [ 263 3156]]
              precision    recall  f1-score   support

           0       0.85      0.58      0.69      2546
           1       0.75      0.92      0.82      3419

    accuracy                           0.78      5965
   macro avg       0.80      0.75      0.76      5965
weighted avg       0.79      0.78      0.77      5965



In [26]:
from lightgbm import LGBMClassifier

X_train_lgbm = X_train.reshape(
    X_train.shape[0],
    -1
)

X_val_lgbm = X_val.reshape(
    X_val.shape[0],
    -1
)

X_test_lgbm = X_test.reshape(
    X_test.shape[0],
    -1
)

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

lgbm.fit(
    X_train_lgbm,
    y_train
)

[LightGBM] [Info] Number of positive: 12726, number of negative: 8558
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.728830 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 236243
[LightGBM] [Info] Number of data points in the train set: 21284, number of used features: 1152
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.597914 -> initscore=0.396781
[LightGBM] [Info] Start training from score 0.396781


LGBMClassifier(learning_rate=0.03, n_estimators=500, random_state=42)

In [27]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_lgbm = CalibratedClassifierCV(
    lgbm,
    method="isotonic",
    cv="prefit"
)

calibrated_lgbm.fit(
    X_val_lgbm,
    y_val
)

/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CalibratedClassifierCV(cv='prefit',
                       estimator=LGBMClassifier(learning_rate=0.03,
                                                n_estimators=500,
                                                random_state=42),
                       method='isotonic')

In [28]:
import shap

background = X_train_lgbm[
    np.random.choice(
        len(X_train_lgbm),
        size=min(100, len(X_train_lgbm)),
        replace=False
    )
]

explainer = shap.TreeExplainer(
    calibrated_lgbm.estimator
)

In [29]:
import json
import joblib
import torch

from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    calibrated_lgbm.predict_proba(X_test_lgbm)[:,1]
)

joblib.dump(
    calibrated_lgbm,
    "gait_model.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

torch.save(
    model.state_dict(),
    "gait_model.pt"
)

metadata = {
    "model_id": "gait_lgbm_v1",
    "validation_auc": float(auc),
    "window_size": WINDOW,
    "stride": STRIDE,
}

with open(
    "metadata.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("Artifacts exported successfully!")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Artifacts exported successfully!
